# `GenVarLoader`

In [143]:
# Automatically reload code in notebook
%load_ext autoreload
%autoreload 2

import pandas as pd
import polars as pl
import genvarloader as gvl
import seqpro as sp
import pooch

pd.set_option("display.max_columns", None)
import sys
sys.path.append("code")
import src.utils as UTILS
import src.pyensembl as PYE
import src.genvarloader as GVL

import os
os.chdir("/grid/koo/home/schilder/projects/GenomeEncoder/data") 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [97]:
mane = PYE.get_mane_transcripts(add_chr=False)
mane.head()

19284 MANE transcripts found.


,#NCBI_GeneID,Ensembl_Gene,HGNC_ID,symbol,name,RefSeq_nuc,RefSeq_prot,Ensembl_nuc,Ensembl_prot,MANE_status,GRCh38_chr,chr_start,chr_end,chr_strand,TranscriptId,chrom
0,GeneID:1,ENSG00000121410.13,HGNC:5,A1BG,alpha-1-B glycoprotein,NM_130786.4,NP_570602.2,ENST00000263100.8,ENSP00000263100.2,MANE Select,NC_000019.10,58345183,58353492,-,ENST00000263100,19
1,GeneID:2,ENSG00000175899.15,HGNC:7,A2M,alpha-2-macroglobulin,NM_000014.6,NP_000005.3,ENST00000318602.12,ENSP00000323929.8,MANE Select,NC_000012.12,9067708,9115919,-,ENST00000318602,12
2,GeneID:9,ENSG00000171428.15,HGNC:7645,NAT1,N-acetyltransferase 1,NM_000662.8,NP_000653.3,ENST00000307719.9,ENSP00000307218.4,MANE Select,NC_000008.11,18210109,18223689,+,ENST00000307719,8
3,GeneID:10,ENSG00000156006.5,HGNC:7646,NAT2,N-acetyltransferase 2,NM_000015.3,NP_000006.2,ENST00000286479.4,ENSP00000286479.3,MANE Select,NC_000008.11,18391282,18401218,+,ENST00000286479,8
4,GeneID:12,ENSG00000196136.18,HGNC:16,SERPINA3,serpin family A member 3,NM_001085.5,NP_001076.2,ENST00000393078.5,ENSP00000376793.3,MANE Select,NC_000014.9,94612391,94624053,+,ENST00000393078,14


In [98]:
bed_path = "mane_transcripts.bed"
mane["score"] = pd.NA
mane.loc[:,['chrom','chr_start','chr_end',"Ensembl_nuc","score","chr_strand"]].to_csv(bed_path, sep="\t", index=False, header=False)

In [99]:
bed = gvl.read_bedlike(bed_path)
bed

chrom,chromStart,chromEnd,name,score,strand
str,i64,i64,str,f64,str
"""19""",58345183,58353492,"""ENST00000263100.8""",null,"""-"""
"""12""",9067708,9115919,"""ENST00000318602.12""",null,"""-"""
"""8""",18210109,18223689,"""ENST00000307719.9""",null,"""+"""
"""8""",18391282,18401218,"""ENST00000286479.4""",null,"""+"""
"""14""",94612391,94624053,"""ENST00000393078.5""",null,"""+"""
…,…,…,…,…,…
"""16""",89317046,89418292,"""ENST00000711617.1""",null,"""-"""
"""2""",10413708,10434222,"""ENST00000649912.2""",null,"""-"""
"""2""",10413708,10434222,"""ENST00000713549.1""",null,"""-"""


In [100]:
vcf_dict = UTILS.list_vcf(dir='/grid/koo/home/schilder/projects/GenomeEncoder/data/1KG/vcf/*.vcf.gz',
                          as_dict=True)
vcf_dict["chr22"]

23 VCF files found.


'/grid/koo/home/schilder/projects/GenomeEncoder/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz'

In [146]:
protein_haplotypes = pooch.retrieve(
    url="doi:10.6084/m9.figshare.6834191.v1/SupplementaryData2-all_protein_haplotypes_GRCh37.fa.gz",
    known_hash="md5:d1a4257bc75314891c5ee502169988d8",
    progressbar=True,
    fname="SupplementaryData2-all_protein_haplotypes_GRCh37.fa.gz",
)

100%|████████████████████████████████████████| 206M/206M [00:00<00:00, 469GB/s]


In [101]:
variants = pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.pgen",
    known_hash="md5:31aba970e35f816701b2b99118dfc2aa",
    progressbar=True,
    fname="1kGP.chr22.pgen",
)

In [102]:
reference = pooch.retrieve(
        url="https://ftp.ensembl.org/pub/release-112/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz",
        known_hash="sha256:974f97ac8ef7ffae971b63b47608feda327403be40c27e391ee4a1a78b800df5",
        progressbar=True,
    )
reference = reference[:-3] + ".bgz"

In [103]:
chrom = "chr22"
save_path = f"gvl/{chrom}.gvl"
bed_chr = bed.filter(pl.col("chrom")==chrom.replace("chr",""))
if not os.path.exists(save_path):
    print("Creating GVL database ==>",save_path)
    gvl.write(
        path=save_path,
        bed=bed_chr,
        variants=variants,#vcf_dict[chrom],
        length=2**15, # <-- required to select sequence subsets afterwards
        max_mem=16*2**30,
        overwrite=False
    )
print("Loading GVL database")
ds = gvl.Dataset.open(save_path, 
                       reference=reference)
ds

Loading GVL database


GVL store chr22.gvl
Is subset: False
# of regions: 423
# of samples: 451
Original region length: 32,768
Max jitter: 0
Has genotypes: True
Has tracks: []

In [80]:
# ref = ds.with_settings(return_sequences="reference")

In [104]:
# [Region, Sample, Phase]
bed_tx = bed_chr[0]
bed_tx

chrom,chromStart,chromEnd,name,score,strand
str,i64,i64,str,f64,str
"""22""",50738204,50745339,"""ENST00000216139.10""",null,"""+"""


In [105]:
regions = ds.get_bed().with_row_index(name="region_idx")
regions_tx = regions.join(bed_tx, how="cross", suffix="_tx")
print(regions_tx)
regions_tx = regions_tx.filter(
    (pl.col("chromStart") <= pl.col("chromEnd_tx")) & 
    (pl.col("chromEnd") >= pl.col("chromStart_tx"))
)
print(regions_tx)

shape: (423, 11)
┌────────────┬───────┬────────────┬──────────┬───┬─────────────┬───────────────┬───────┬───────────┐
│ region_idx ┆ chrom ┆ chromStart ┆ chromEnd ┆ … ┆ chromEnd_tx ┆ name          ┆ score ┆ strand_tx │
│ ---        ┆ ---   ┆ ---        ┆ ---      ┆   ┆ ---         ┆ ---           ┆ ---   ┆ ---       │
│ u32        ┆ str   ┆ i64        ┆ i64      ┆   ┆ i64         ┆ str           ┆ f64   ┆ str       │
╞════════════╪═══════╪════════════╪══════════╪═══╪═════════════╪═══════════════╪═══════╪═══════════╡
│ 0          ┆ 22    ┆ 15512281   ┆ 15545049 ┆ … ┆ 50745339    ┆ ENST000002161 ┆ null  ┆ +         │
│            ┆       ┆            ┆          ┆   ┆             ┆ 39.10         ┆       ┆           │
│ 1          ┆ 22    ┆ 15689444   ┆ 15722212 ┆ … ┆ 50745339    ┆ ENST000002161 ┆ null  ┆ +         │
│            ┆       ┆            ┆          ┆   ┆             ┆ 39.10         ┆       ┆           │
│ 2          ┆ 22    ┆ 16575396   ┆ 16608164 ┆ … ┆ 50745339    ┆ ENST00000

In [106]:
seq_block = ds.isel(regions=regions_tx['region_idx'], samples=0)
seq_block

array([[[b'C', b'T', b'T', ..., b'G', b'T', b'T'],
        [b'C', b'T', b'T', ..., b'T', b'C', b'T']]], dtype='|S1')

In [107]:
tx_len = regions_tx['chromEnd_tx'][0] - regions_tx['chromStart_tx'][0]
tx_start = regions_tx['chromStart_tx'][0] - regions_tx['chromStart'][0]
tx_end = regions_tx['chromEnd_tx'][0] - regions_tx['chromStart'][0] 
tx_start, tx_end, tx_len

(12817, 19952, 7135)

In [108]:
seqs = seq_block[:,:,tx_start:tx_end]
print(seqs.shape)
print("sequence length matches:",seqs.shape[2]==tx_len)
seqs

(1, 2, 7135)
sequence length matches: True


array([[[b'A', b'C', b'T', ..., b'A', b'G', b'A'],
        [b'C', b'T', b'A', ..., b'C', b'A', b'A']]], dtype='|S1')

In [109]:
tx = PYE.get_transcript(regions_tx['name'][0].split(".")[0])

In [110]:
start = tx.first_start_codon_spliced_offset
end = tx.last_stop_codon_spliced_offset
print(start, end)
seqs_spliced = seqs[:,:,start:end+1]

print(seqs_spliced.shape)
print("Sequence divisible by 3:",seqs_spliced.shape[2]%3==0)
seqs_spliced

32 1297
(1, 2, 1266)
Sequence divisible by 3: True


array([[[b'A', b'T', b'G', ..., b'A', b'G', b'G'],
        [b'T', b'G', b'G', ..., b'G', b'G', b'T']]], dtype='|S1')

In [68]:
print(len(tx.coding_sequence), seqs_spliced.shape[2])
print("Sequence is the expected length:", seqs_spliced.shape[2]==len(tx.coding_sequence))

1266 1266
Sequence is the expected length: True


In [120]:
seqs_spliced[:,0,:].tobytes(), seqs_spliced[:,1,:].tobytes()

(b'ATGGTTGAGATGCTACCAACTGCCATTCTGCTGGTCTTGGCAGTGTCCGTGGTTGCTAAAGATAACGCCACGTGTGAGTAAGTGTCGGGGCACCTTGGTGGGGGAAGGATCTTCTGAGGAGCAGGTACCACCCCGACTCCCTCTGTCCAGGGCTAGGGAAAAGGAGGCTGCATCCCTAACCTGGACCCCCCCTGCTCCCAGAATCAGCAGCCTGGAGCCCCCAGACCCTCAGCTTTCGTGGTTTCCTCCAGAGATGGACCCCTCAGCACCTCAGGCTCCTTGTGCCTCTCCCACTCCCCCAGGGACTGACCCCACTGTCTTGAAGACATGAAGTCCTGATTTTGGGAGCCCTTATCCCCCCACAGACAGCTGTCCCAACCCGTGGTTGCCCCCAACAGCCCCAGGATATCATCGCTTCACACCGCTTGCACCCCTACCCCCCAGTAGGCTCTCTCACTCCAAGGTACCCCGAAATACCAACACCTCCCAAGCTATATGTGGCCTCCCACCCGTGACACAGTTCCCAGAGCCTCCACCTCTAGACCTCCACTGCTCTCAGTGTGCCCCCTACACCTGTGGGCCACAGTATCTGCCCCTGGCTGCTATCCCTCCTCCCATCACTGTCAACGACCCCCTTCATCACCTGACTTCCCTGAGTCTCCCACCCAAGATTGGTTATAAGGACCTCAGGCCATTACACCCCTCTGTCCCCAGGCCCCGCATCCCCACCTCTACCCTCCTGTTCTGCCCAGGGACGGGCCATCCCTCAGGGCCCATGCAGCCTGTCCTGGCTTCCTATGGCCTCCTCTTTCTCCATCTGTGACTGCACCCACAAGACCTGAGAAGTCGTGGCCCCAGAACCATTTCCTAGAGCCTGCGGCTTCCTACATAGCGCAGGCTGCCCCTGCTTTCCCAGAACCCGGAAGCTCTTCCCCACTTTTCCCAACCCCATGTCCCTGCCTCCCCTCAGTTGTGGAGTTACAAGGACAGGCTGTGC

In [123]:
seqs_spliced[:,0,:].tobytes().decode()

'ATGGTTGAGATGCTACCAACTGCCATTCTGCTGGTCTTGGCAGTGTCCGTGGTTGCTAAAGATAACGCCACGTGTGAGTAAGTGTCGGGGCACCTTGGTGGGGGAAGGATCTTCTGAGGAGCAGGTACCACCCCGACTCCCTCTGTCCAGGGCTAGGGAAAAGGAGGCTGCATCCCTAACCTGGACCCCCCCTGCTCCCAGAATCAGCAGCCTGGAGCCCCCAGACCCTCAGCTTTCGTGGTTTCCTCCAGAGATGGACCCCTCAGCACCTCAGGCTCCTTGTGCCTCTCCCACTCCCCCAGGGACTGACCCCACTGTCTTGAAGACATGAAGTCCTGATTTTGGGAGCCCTTATCCCCCCACAGACAGCTGTCCCAACCCGTGGTTGCCCCCAACAGCCCCAGGATATCATCGCTTCACACCGCTTGCACCCCTACCCCCCAGTAGGCTCTCTCACTCCAAGGTACCCCGAAATACCAACACCTCCCAAGCTATATGTGGCCTCCCACCCGTGACACAGTTCCCAGAGCCTCCACCTCTAGACCTCCACTGCTCTCAGTGTGCCCCCTACACCTGTGGGCCACAGTATCTGCCCCTGGCTGCTATCCCTCCTCCCATCACTGTCAACGACCCCCTTCATCACCTGACTTCCCTGAGTCTCCCACCCAAGATTGGTTATAAGGACCTCAGGCCATTACACCCCTCTGTCCCCAGGCCCCGCATCCCCACCTCTACCCTCCTGTTCTGCCCAGGGACGGGCCATCCCTCAGGGCCCATGCAGCCTGTCCTGGCTTCCTATGGCCTCCTCTTTCTCCATCTGTGACTGCACCCACAAGACCTGAGAAGTCGTGGCCCCAGAACCATTTCCTAGAGCCTGCGGCTTCCTACATAGCGCAGGCTGCCCCTGCTTTCCCAGAACCCGGAAGCTCTTCCCCACTTTTCCCAACCCCATGTCCCTGCCTCCCCTCAGTTGTGGAGTTACAAGGACAGGCTGTGCTC

In [130]:
len(seqs_spliced)

1

In [135]:
def bytearray_to_bioseq(byte_arr):
    from Bio.Seq import Seq
   
    # Sample, Ploid, Sequence
    if byte_arr.ndim == 1:
        return [Seq(byte_arr.tobytes().decode())]
    elif byte_arr.ndim == 2:
        ploid_idx = range(byte_arr.shape[-2])
        return [Seq(byte_arr[idx,:].tobytes().decode()) for idx in ploid_idx]
    elif byte_arr.ndim == 3:
        ploid_idx = range(byte_arr.shape[-2])
        return [Seq(byte_arr[:,idx,:].tobytes().decode()) for idx in ploid_idx]
    else:
        raise ValueError(f"Invalid number of dimensions: {byte_arr.ndim}")


In [140]:
bioseqs = bytearray_to_bioseq(seqs_spliced)
bioseqs

[Seq('ATGGTTGAGATGCTACCAACTGCCATTCTGCTGGTCTTGGCAGTGTCCGTGGTT...AGG'),
 Seq('TGGTTGAGATGCTACCAACTGCCATTCTGCTGGTCTTGGCAGTGTCCGTGGTTG...GGT')]

In [138]:
aa_bioseqs = [x.translate() for x in bioseqs]
aa_bioseqs

[Seq('MVEMLPTAILLVLAVSVVAKDNATCE*VSGHLGGGRIF*GAGTTPTPSVQG*GK...*GR'),
 Seq('WLRCYQLPFCWSWQCPWLLKITPRVSKCRGTLVGEGSSEEQVPPRLPLSRAREK...EGG')]

In [72]:
ref_aa_pyensembl = tx.protein_sequence
print(ref_aa_pyensembl)
sp.AA.translate(tx.coding_sequence, length_axis=-1)


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /grid/koo/home/schilder/.cache/pyensembl/GRCh38/ensembl111/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


MVEMLPTAILLVLAVSVVAKDNATCDGPCGLRFRQNPQGGVRIVGGKAAQHGAWPWMVSLQIFTYNSHRYHTCGGSLLNSRWVLTAAHCFVGKNNVHDWRLVFGAKEITYGNNKPVKAPLQERYVEKIIIHEKYNSATEGNDIALVEITPPISCGRFIGPGCLPHFKAGLPRGSQSCWVAGWGYIEEKAPRPSSILMEARVDLIDLDLCNSTQWYNGRVQPTNVCAGYPVGKIDTCQGDSGGPLMCKDSKESAYVVVGITSWGVGCARAKRPGIYTATWPYLNWIASKIGSNALRMIQSATPPPPTTRPPPIRPPFSHPISAHLPWYFQPPPRPLPPRPPAAQPRPPPSPPPPPPPPASPLPPPPPPPPPTPSSTTKLPQGLSFAKRLQQLIEVLKGKTYSDGKNHYDMETTELPELTSTS


ValueError: gufunc_translate: Input operand 2 has a mismatch in its core dimension 0, with gufunc signature (k),(j,k),(j)->() (size 21 is different from 64)